In [1]:
! pip install py-feat

INFO: pip is looking at multiple versions of scikit-image to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 88.4 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.7/38.7 MB 47.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 85.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 381.4/381.4 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 93.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 92.0 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: numexpr
    Found existing installation: numexpr 2.10.2
    Uninstalling numexpr-2.10.2:
      Successfully uninstalled numexpr-2.10.2
  Attempting uninst

In [2]:
from feat import Detector
detector = Detector(
    face_model="retinaface",
    landmark_model="mobilefacenet",
    au_model='xgb',
    emotion_model="resmasknet",
    facepose_model="img2pose",
    device='cuda'
)
detector

100%|██████████| 1.79M/1.79M [00:00<00:00, 56.9MB/s]
100%|██████████| 12.3M/12.3M [00:00<00:00, 164MB/s]
100%|██████████| 966k/966k [00:00<00:00, 34.7MB/s]
100%|██████████| 33.6M/33.6M [00:00<00:00, 292MB/s]
100%|██████████| 130k/130k [00:00<00:00, 12.1MB/s]
100%|██████████| 45.9M/45.9M [00:00<00:00, 310MB/s]
100%|██████████| 130k/130k [00:00<00:00, 12.3MB/s]
100%|██████████| 53.9M/53.9M [00:00<00:00, 387MB/s]
100%|██████████| 130k/130k [00:00<00:00, 8.97MB/s]
100%|██████████| 167k/167k [00:00<00:00, 9.89MB/s]
100%|██████████| 531k/531k [00:00<00:00, 22.7MB/s]
100%|██████████| 494k/494k [00:00<00:00, 19.5MB/s]
100%|██████████| 207k/207k [00:00<00:00, 13.1MB/s]
100%|██████████| 1.15M/1.15M [00:00<00:00, 35.2MB/s]
100%|██████████| 572k/572k [00:00<00:00, 22.8MB/s]
100%|██████████| 330k/330k [00:00<00:00, 14.7MB/s]
100%|██████████| 335k/335k [00:00<00:00, 16.3MB/s]
100%|██████████| 587k/587k [00:00<00:00, 22.9MB/s]
100%|██████████| 207k/207k [00:00<00:00, 11.1MB/s]
100%|██████████| 690k/6

feat.detector.Detector(face_model=retinaface, landmark_model=mobilefacenet, au_model=xgb, emotion_model=resmasknet, facepose_model=img2pose, identity_model=facenet)

In [4]:
import cv2
from feat import Detector
import numpy as np
import pandas as pd
import os

# Initialize the py-feat detector
# detector = Detector(
#     face_model="retinaface",
#     landmark_model="mobilefacenet",
#     au_model="svm",
#     emotion_model="resmasknet",
#     facepose_model="img2pose",
# )

# Path to the video file
video_path = "/kaggle/input/zain-data/3bvh.mp4"
# Open the video file
cap = cv2.VideoCapture(video_path)

# Check if the video is successfully opened
if not cap.isOpened():
    print("Error: Unable to open video file.")
    exit()

# Directory to temporarily save frames
temp_frame_dir = "temp_frames 2"
os.makedirs(temp_frame_dir, exist_ok=True)

# List to store AU data for each frame
au_data = []

# Define the number of frames you want to process (e.g., the first 10 frames)
frame_limit = 10 # Change this number to the desired number of frames

# Loop through the frames of the video
frame_count = 0
while True:
    ret, frame = cap.read()

    # Break the loop if no frame is returned or if we've processed the desired number of frames
    if not ret:
        break

    frame_count += 1

    # Save the frame as an image file
    temp_frame_path = os.path.join(temp_frame_dir, f"frame_{frame_count}.jpg")
    cv2.imwrite(temp_frame_path, frame)  # Save frame as JPEG image

    # Detect facial features and action units in the frame (pass image path)
    detections = detector.detect_image([temp_frame_path])  # Pass the image path

    # Extract action units (AUs) as a DataFrame for this frame
    frame_aus = detections.aus  # AUs as DataFrame (or array-like)

    # Flatten the AU data into a 1D array (if necessary)
    frame_aus_flat = frame_aus.values.flatten()  # Use .values to convert to numpy

    # Include the frame number as an additional column
    frame_aus_flat = np.append(frame_aus_flat, frame_count)

    # Append flattened AU data to the list
    au_data.append(frame_aus_flat)  # Each row will contain AU values and the frame number

    # Optionally, remove the temporary image after processing
    os.remove(temp_frame_path)

# Release video capture
cap.release()
# AU labels

au_labels = [
    "Inner Brow Raise (AU1)", "Outer Brow Raise (AU2)", "Brow Lowerer (AU4)", "Upper Lid Raiser (AU5)", 
    "Cheek Raiser (AU6)", "Nose Wrinkler (AU9)", "Upper Lip Raiser (AU10)", "Lip Corner Puller (AU12)", 
    "Dimpler (AU14)", "Lip Corner Depressor (AU15)", "Chin Raiser (AU17)", "Lip Stretcher (AU20)", 
    "Lip Tightener (AU23)", "Lip Pressor (AU24)", "Lips Part (AU25)", "Jaw Drop (AU26)", 
    "Mouth Stretch (AU27)", "Lip Suck (AU28)", "Blink (AU45)", "Head Turn Left", 
    "Head Turn Right", "Head Up", "Head Down", "Head Tilt Left", "Head Tilt Right", 
    "Eyes Turn Left", "Eyes Turn Right", "Eyes Up", "Eyes Down", "Smile", 
    "Frown", "Pucker Lips", "Raise Cheek", "Cheek Suck", "Nostril Flare", 
    "Nostril Compress", "Squint", "Wide Eyes", "Brow Raise", "Eye Closure"
]


# Determine the number of columns in the first row of au_data
num_features = len(au_data[0]) - 1  # Subtract 1 for the frame number

# Dynamically adjust au_labels to match the number of features
if num_features > len(au_labels):
    # If there are more features than labels, add generic labels
    au_labels += [f"Feature_{i}" for i in range(len(au_labels) + 1, num_features + 1)]
elif num_features < len(au_labels):
    # If there are fewer features than labels, truncate the labels
    au_labels = au_labels[:num_features]

# Add the "frame" column to the labels
column_names = au_labels + ["frame"]

# Validate au_data consistency
for i, row in enumerate(au_data):
    if len(row) != len(column_names):
        # print(f"Warning: Row {i} has inconsistent length. Expected {len(column_names)}, got {len(row)}.")
        # print(row)  # Debugging info for mismatched rows
        # Ensure row matches the expected length
        au_data[i] = np.resize(row, len(column_names))

# Create the DataFrame
au_df = pd.DataFrame(au_data, columns=column_names)

# Save to a CSV file (optional)
au_df.to_csv('sev_bvh3.csv', index=False)

print("Processing completed.")

100%|██████████| 1/1 [00:01<00:00,  1.24s/it]


Processing completed.
